In [3]:
# wrong approach to append lists
nums1 = [1,2] 
nums2 = [3,4]
new_list = []

new_list.append(nums1)
new_list.append(nums2)

print(new_list)

[[1, 2], [3, 4]]


In [ ]:
# correct approach
nums1 = [1,2] 
nums2 = [3,4]


nums1.extend(nums2)


print(nums1)

[1, 2, 3, 4]


In [ ]:
# correct approach
nums1 = [1,2] 
nums2 = [3,4]


nums1 += (nums2)

"""
nums = nums1 + nums2 is also fine

"""


print(nums1)

[1, 2, 3, 4]


In [ ]:
#Complexity of solution
#Time:	O((m+n)log(m+n))      # required: O(log (m+n))
#Space:	O(m+n)

def findMedianSortedArrays(nums1: list[int], nums2: list[int]) -> float:
  
    nums1 += (nums2)

    nums = sorted(nums1)

    n = len(nums)

    if n % 2 == 0:
        middle1 = nums[(n // 2) - 1]
        middle2 = nums[ n // 2]

        return (middle1 + middle2) / 2.0
   
    else:
        return nums[n // 2]


In [ ]:
#time complexity:O(log(min(n,m)))	
#space complexity:O(1)
def findMedianSortedArrays(nums1: list[int], nums2: list[int]) -> float:

    if len(nums1) > len(nums2):
        nums1,nums2 = nums2,nums1

    n = len(nums1)
    m = len(nums2)


    low = 0
    high = n # we are calculating partition for smaller array

    # Number of elements required on the left side
    left_size = (n + m + 1) // 2

    while low <= high:

        # Partition position in nums1
        i = low + (high - low) // 2

        # Remaining elements needed from nums2
        j = left_size  - i

        # Values immediately around nums1 partition |
        left1 = nums1[i - 1] if i > 0 else float("-inf")
        right1 = nums1[i] if i < n else float("inf")

        # Values immediately around nums2 partition |
        left2 = nums2[j - 1] if j > 0 else float("-inf")
        right2 = nums2[j] if j < m else float("inf")

        # Correct partition
        if left1 <= right2 and left2 <= right1:

            # if total number of elements are even
            if (n + m) % 2 == 0:
                left_max = max(left1, left2)
                right_min = min(right1, right2)

                return (left_max + right_min) / 2

            # if total number of elements are odd
            else:
                return max(left1, left2)

        # Too many elements taken from nums1
        elif left1 > right2:
            high = i - 1

        # Too few elements taken from nums1
        else:
            low = i + 1

In [27]:
nums1 = [1,2] 
nums2 = [3,4]

iret = findMedianSortedArrays(nums1,nums2)
print(iret)

2.5


# LeetCode 4 — Median of Two Sorted Arrays

## 1. Core Idea

We are given two sorted arrays and need to find their median.

The optimal solution does **not merge the arrays**.

Instead, we imagine putting a partition (an invisible wall) in each array:


nums1 = [ ... LEFT ... | ... RIGHT ... ]
                         ^
                         i

nums2 = [ ... LEFT ... | ... RIGHT ... ]
                         ^
                         j


We use **binary search** to find the correct position of the partition.


## 2. Brute-Force Approach

A simple solution is:

nums1 += nums2
nums = sorted(nums1)

If the total number of elements is:

N = n + m

then sorting takes:

O(N log N)

So:

Time  = O((n + m) log(n + m))
Space = O(n + m)


The optimal solution improves this to:

Time  = O(log(min(n, m)))
Space = O(1)


---

# 3. Why Search the Smaller Array?

First make `nums1` the smaller array:

if len(nums1) > len(nums2):
    nums1, nums2 = nums2, nums1

Then:

n = len(nums1)
m = len(nums2)


Binary search is performed only on `nums1`.

Therefore the time complexity is:
O(log(min(n, m)))


---

# 4. What Are `i` and `j`?

`i` and `j` are **partition positions**, not values.

Example:

nums1 = [1, 3 | 8]
              ^
              i = 2

This means 2 elements are on the left.

Similarly:

nums2 = [2, 4 | 9, 10]
              ^
              j = 2


This means 2 elements are on the left.

We do **not** physically split the arrays.

The partition is just an imaginary wall represented by an index.

---

# 5. How Do We Find `i`?

Binary search chooses a candidate partition:

i = low + (high - low) // 2

Initially:


low = 0
high = n

Possible partition positions are:
0, 1, 2, ..., n


After choosing `i`, the corresponding partition in `nums2` is forced.

---

# 6. How Do We Find `j`?

We want approximately half of all elements to be on the left.

Calculate:

left_size = (n + m + 1) // 2


Then:

j = left_size - i


Why?

Because:

elements from nums1 on left
+
elements from nums2 on left
=
total elements required on left


Therefore:

i + j = left_size

and:

j = left_size - i


### Example

If:

n = 3
m = 4

then:

left_size = (3 + 4 + 1) // 2
          = 4


If:

i = 2

then:

j = 4 - 2
  = 2

So:

nums1 = [1, 3 | 8]
nums2 = [2, 4 | 9, 10]

There are:

2 + 2 = 4

elements on the left.

---

# 7. The Four Important Boundary Values

Once `i` and `j` are known, look at the elements immediately around the partitions.

For `nums1`:

left1 = nums1[i - 1] if i > 0 else float("-inf")
right1 = nums1[i] if i < n else float("inf")

For `nums2`:

left2 = nums2[j - 1] if j > 0 else float("-inf")
right2 = nums2[j] if j < m else float("inf")


Visual:

nums1 = [ ... left1 | right1 ... ]
                    i

nums2 = [ ... left2 | right2 ... ]
                    j


---

# 8. Why `-inf` and `inf`?

Sometimes the partition is at an edge.

### Partition at beginning

nums1 = [ | 1, 3, 8]


There is no left element, so:

left1 = float("-inf")


### Partition at end

nums1 = [1, 3, 8 | ]


There is no right element, so:

right1 = float("inf")


This lets us use the same comparison logic for edge cases.

---

# 9. How Do We Know the Partition Is Correct?

This is the key condition:

left1 <= right2 and left2 <= right1


Why?

We want every element on the left side to be less than or equal to every element on the right side.

Because both arrays are already sorted, we only need to check the two cross-boundary comparisons:

left1 <= right2
left2 <= right1


If both are true, the partition is correct.

---

# 10. Example of a Correct Partition

Suppose:

nums1 = [1, 3 | 8]
nums2 = [2, 4 | 9, 10]


Then:

left1  = 3
right1 = 8

left2  = 4
right2 = 9

Check:

3 <= 9   ✓
4 <= 8   ✓


So the partition is correct.

---

# 11. What If the Partition Is Wrong?

There are two cases.

## Case A: `left1 > right2`

Example:

left1  = 8
right2 = 5

So:

8 > 5

We took too many elements from `nums1` into the left half.

Move the partition in `nums1` to the **left**:

high = i - 1

Visual:

Before:

nums1 = [1, 3, 8 | 10]
               ^
               i

Move left:

nums1 = [1, 3 | 8, 10]
            ^

---

## Case B: `left2 > right1`

Example:

left2  = 8
right1 = 4

So:

8 > 4


We did not take enough elements from `nums1` into the left half.

Move the partition in `nums1` to the **right**:

low = i + 1


Visual:

Before:

nums1 = [1 | 3, 4]
          ^
          i

Move right:

nums1 = [1, 3 | 4]
             ^

---

# 12. Finding the Median

Once the partition is correct:

left1 <= right2 and left2 <= right1


the largest element on the left is:

max(left1, left2)

and the smallest element on the right is:

min(right1, right2)


These give us the middle value(s).

## Odd Total Length

Example:

[1, 2, 3, 4, 5]

Median:

3


So:


median = max(left1, left2)

## Even Total Length

Example:

[1, 2, 3, 4]

Middle values:

2 and 3

So:

median = (max(left1, left2) + min(right1, right2)) / 2

---

# 13. Complexity

Let:

n = len(nums1)
m = len(nums2)


We binary-search only the smaller array.

### Time Complexity

Binary search takes:

O(log(min(n, m)))

Therefore:

TIME = O(log(min(n, m)))


### Space Complexity

We do not create a merged array or any extra data structure.

Only a few variables are used:


low, high, i, j
left1, right1, left2, right2


Therefore:

SPACE = O(1)

---

# 14. Brute Force vs Optimal

|   Approach              |       Time           |     Space |
| Merge + Sort            | O((n+m) log(n+m))    | O(n+m)    |
| Binary Search Partition | **O(log(min(n,m)))** | **O(1)**  |

---

# 15. Quick Revision Cheat Sheet

Remember these 7 points:

### 1. Search the smaller array

if len(nums1) > len(nums2):
    nums1, nums2 = nums2, nums1

### 2. Choose partition `i`


i = low + (high - low) // 2

### 3. Calculate partition `j`

j = left_size - i

### 4. Get four boundary values

left1, right1
left2, right2

### 5. Check the partition

left1 <= right2 and left2 <= right1

### 6. Move the partition if necessary

left1 > right2
→ move i LEFT

left2 > right1
→ move i RIGHT

### 7. Complexity


Time  = O(log(min(n, m)))
Space = O(1)

---

# 16. Most Important Mental Picture

When revising this problem, draw:


nums1 = [ LEFT | RIGHT ]
              ^
              i

nums2 = [ LEFT | RIGHT ]
              ^
              j

             LEFT HALF | RIGHT HALF


The goal is:

1. Put the correct number of elements on the LEFT.
2. Make sure the LEFT side is completely <= the RIGHT side.
3. Use the middle boundary values to calculate the median.


**Important:** The arrays are never actually split.

`i` and `j` are simply imaginary walls represented by indices.

---

# 18. One-Line Memory Trick

> **Binary-search the smaller array, choose `i`, calculate `j`, check the four boundary values, adjust the partition until the left side is valid, then calculate the median.**


In [ ]:
# alternative approach :Two pointer merge approach
def findMedianSortedArrays(nums1: list[int], nums2: list[int]) -> float:

    n = len(nums1)
    m = len(nums2)
    total = n + m

    i = 0
    j = 0

    previous = 0
    current = 0

    for _ in range(total // 2 + 1):

        previous = current

        if i < n and (j >= m or nums1[i] <= nums2[j]):
            current = nums1[i]
            i += 1
        else:
            current = nums2[j]
            j += 1

    if total % 2 == 1:
        return current

    return (previous + current) / 2

1. Two-Pointer Merge — O(n + m) time, O(1) space

The idea is exactly like the merge step of Merge Sort.

We have two sorted arrays:

nums1 = [1, 3, 8]
nums2 = [2, 4, 9]

We compare the current elements:

1 vs 2 → take 1
3 vs 2 → take 2
3 vs 4 → take 3
8 vs 4 → take 4

We only need to go until the middle; we don't actually need to create the merged array.

------
Example:
nums1 = [1, 3]
nums2 = [2, 4]

Merged conceptually:

1 → 2 → 3 → 4

We stop after reaching the two middle elements:

previous = 2
current  = 3

Therefore:

(2 + 3) / 2 = 2.5

More precisely, we only traverse up to the middle, so the actual work is O((n+m)/2), which simplifies to O(n+m).